In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
from analyzer import Analyzer
analyzer = Analyzer()

Analyzer opened


In [3]:
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
from pypfopt.hierarchical_portfolio import HRPOpt

In [4]:
df_for_date = pd.read_csv('../csv_data/stocks/AAPL.csv')

df_pypfopt = pd.DataFrame(
    {
        'Date':df_for_date['Date']
    }
)

# df = pd.read_csv('../csv_data/stocks/AAPL.csv')

stock_lists = [
            'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META', 'NVDA', 'NFLX', 'AMD',
            'DIS', 'BABA', 'V', 'JPM', 'JNJ', 'WMT', 'PG', 'MA', 'UNH', 'HD',
            'BAC', 'ADBE', 'CRM', 'PYPL', 'INTC', 'VZ', 'KO', 'PEP', 'T',
            'CSCO', 'XOM', 'ABT', 'TMO', 'ACN', 'TXN', 'QCOM', 'COST', 'SBUX',
            'MDT', 'LOW', 'HON', 'UPS', 'IBM', 'GS', 'CAT', 'MMM', 'SLB', 'SPY',
            'LLY', 'ASML', 'MU', 'CVX', 'NVS', 'PM', 'NVO', 'SHEL',
            'KLAC', 'PFE' ,'SPOT', 'BX', 'HOOD', 'GM', 'SE', 'TLT', 'SMH'        ]

for s in stock_lists:
    df = pd.read_csv('../csv_data/stocks/'+s+'.csv')
    df.rename(columns={'Close':s}, inplace=True)
    df_pypfopt = pd.merge(df_pypfopt, df[['Date', s]], on='Date', how='inner')



In [5]:
df_pypfopt.head()

,Date,AAPL,MSFT,GOOGL,AMZN,TSLA,META,NVDA,NFLX,AMD,...,SHEL,KLAC,PFE,SPOT,BX,HOOD,GM,SE,TLT,SMH
0,2021-10-20 00:00:00-04:00,145.583878,295.029755,140.432693,170.753006,288.600006,337.547089,22.006073,62.514000,116.389999,...,41.087818,31.531954,32.849895,251.179993,108.057930,41.830002,55.276054,360.600006,119.322807,129.287918
1,2021-10-21 00:00:00-04:00,145.798462,298.244843,140.548569,171.750504,298.000000,338.636597,22.592491,65.316002,119.330002,...,40.597485,31.958693,32.895950,255.000000,111.273903,41.279999,55.985336,363.299988,119.180641,130.762726
2,2021-10-22 00:00:00-04:00,145.027954,296.709229,136.269806,166.777496,303.226654,321.530426,22.626343,66.477997,119.820000,...,40.464527,32.675003,33.126202,252.960007,115.564621,39.590000,55.371899,355.299988,120.493225,129.608078
3,2021-10-25 00:00:00-04:00,144.979202,295.720825,136.151443,166.018494,341.619995,325.571777,23.064417,67.166000,122.360001,...,40.772015,32.694054,33.118526,257.890015,117.193604,39.009998,55.362309,348.320007,120.309319,130.263031
4,2021-10-26 00:00:00-04:00,145.642441,297.621033,137.995361,168.803497,339.476654,312.813965,24.608610,66.851997,122.930000,...,40.605801,32.574032,33.433208,252.199997,119.368362,39.570000,54.988510,343.570007,121.304077,131.005295


### 12 - 3 - 12
#### 12 months data train - 3 months data test - difference per test: 12 months

In [6]:
# # delta = 1 year (12 month), 1 array has difference of 3 months
start_end_dates = [
    ('2021-07-01', '2022-07-01'),
    ('2021-10-01', '2022-10-01'),
    ('2022-01-01', '2023-01-01'),
    ('2022-04-01', '2023-04-01'),
    ('2022-07-01', '2023-07-01'),
    ('2022-10-01', '2023-10-01'),
    ('2023-01-01', '2024-01-01'),
    ('2023-04-01', '2024-04-01'),
    ('2023-07-01', '2024-07-01'),
    ('2023-10-01', '2024-10-01'),
    ('2024-01-01', '2025-01-01'),
    ('2024-04-01', '2025-04-01'),
    ('2024-07-01', '2025-07-01'),
    ('2024-10-01', '2025-10-01'),
]

In [7]:
all_weight_data = {}

for (start_date, end_date) in start_end_dates:
    df_semester =  df_pypfopt[(df_pypfopt['Date'] >= start_date) & (df_pypfopt['Date'] <= end_date)].reset_index(drop=True)
    if 'Date' in df_semester.columns:
        df_semester.set_index('Date', inplace=True)
        
    mu = expected_returns.mean_historical_return(df_semester)
    S = risk_models.sample_cov(df_semester)
    
    # Optimize for maximal Sharpe ratio
    ef = EfficientFrontier(mu, S)
    ef.add_constraint(lambda x : x <= 0.20)
    weights = ef.max_sharpe(risk_free_rate=0.05)
    # print(weights)
    res = ef.portfolio_performance(risk_free_rate=0.05, verbose=True)
    expected_annual_return = str(float(res[0]) * 100)+'%'
    annual_volatility = str(float(res[1]) * 100)+'%'
    sharpe_ratio = float(res[2])

    weights_to_buy = {k:weights[k]*100 for k, v in sorted([(v,weights[v]*100) for k,v in enumerate(weights) if weights[v]>0], key=lambda x: x[1], reverse=True)}
    all_weight_data[start_date+'_'+end_date] = {
        'expected_annual_return' : expected_annual_return,
        'annual_volatility' : annual_volatility,
        'sharpe_ratio' : sharpe_ratio,
        'weights_to_buy': weights_to_buy
    }
    print('======')

Expected annual return: 45.5%
Annual volatility: 19.2%
Sharpe Ratio: 2.11
Expected annual return: 28.3%
Annual volatility: 19.3%
Sharpe Ratio: 1.21
Expected annual return: 48.9%
Annual volatility: 22.7%
Sharpe Ratio: 1.94
Expected annual return: 24.8%
Annual volatility: 17.7%
Sharpe Ratio: 1.12
Expected annual return: 92.6%
Annual volatility: 23.1%
Sharpe Ratio: 3.79
Expected annual return: 104.7%
Annual volatility: 19.3%
Sharpe Ratio: 5.18
Expected annual return: 130.1%
Annual volatility: 20.3%
Sharpe Ratio: 6.17
Expected annual return: 97.9%
Annual volatility: 14.4%
Sharpe Ratio: 6.45
Expected annual return: 78.2%
Annual volatility: 13.0%
Sharpe Ratio: 5.66
Expected annual return: 76.2%
Annual volatility: 11.1%
Sharpe Ratio: 6.39
Expected annual return: 93.3%
Annual volatility: 13.1%
Sharpe Ratio: 6.76
Expected annual return: 81.8%
Annual volatility: 13.4%
Sharpe Ratio: 5.72
Expected annual return: 113.2%
Annual volatility: 20.2%
Sharpe Ratio: 5.35
Expected annual return: 130.4%
Annu

In [8]:
all_weight_data

{'2021-07-01_2022-07-01': {'expected_annual_return': '45.50520454203638%',
  'annual_volatility': '19.19015426258591%',
  'sharpe_ratio': 2.1107284489634024,
  'weights_to_buy': {'KO': 20.0,
   'XOM': 20.0,
   'LLY': 20.0,
   'CVX': 20.0,
   'PFE': 16.205332488018538,
   'UNH': 3.79466751198151}},
 '2021-10-01_2022-10-01': {'expected_annual_return': '28.32605578319206%',
  'annual_volatility': '19.27470254786263%',
  'sharpe_ratio': 1.210190181937655,
  'weights_to_buy': {'UNH': 20.0,
   'XOM': 20.0,
   'LLY': 20.0,
   'CVX': 20.0,
   'KO': 17.88040829205897,
   'PFE': 1.23286760372992,
   'JNJ': 0.88672410421108}},
 '2022-01-01_2023-01-01': {'expected_annual_return': '48.91114768926181%',
  'annual_volatility': '22.68787995744659%',
  'sharpe_ratio': 1.9354451703562254,
  'weights_to_buy': {'XOM': 20.0,
   'LLY': 20.0,
   'CVX': 20.0,
   'NVO': 19.05829678733679,
   'SLB': 10.91613234730259,
   'KO': 5.9106916338841495,
   'PM': 4.11487923147648}},
 '2022-04-01_2023-04-01': {'expected

## Backtest

In [9]:
from backtest import ManualBacktest
from datetime import datetime

In [10]:
backtest_months = 12

In [11]:
mb = ManualBacktest()

for a in all_weight_data:
    allocation = all_weight_data[a]['weights_to_buy']
    print('allocation : ')
    print(allocation)

    start_date_str = a.split('_')[1]
    start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = mb.delta_date(start_date, months = backtest_months)
    end_date_str = datetime.strftime(mb.delta_date(start_date, months = backtest_months), '%Y-%m-%d')

    print('start_date_str : ', start_date_str)
    print('end_date_str : ', end_date_str)

    df_port = mb.make_portfolio_rebalance(start_date_str, end_date_str, allocation)

    return_backtest = mb.calculate_return_df(df_port, start_date_str, end_date_str)
    std_backtest = mb.calculate_std_df(df_port, start_date_str, end_date_str)
    sharpe_ratio_backtest = return_backtest / std_backtest

    return_spy = mb.calculate_return_benchmark(start_date_str, end_date_str, 'SPY', 'Date')
    std_spy = mb.calculate_std_benchmark(start_date_str, end_date_str, 'SPY', 'Date')
    sharpe_ratio_spy = return_spy / std_spy
    
    return_1n = mb.calculate_return_1n(allocation, df_port)
    
    print('return_backtest : ', return_backtest)
    print('std_backtest : ', std_backtest)

    all_weight_data[a]['backtest'] = {}

    all_weight_data[a]['backtest']['start_date_backtest'] = start_date_str
    all_weight_data[a]['backtest']['end_date_backtest'] = str(end_date)[0:10]
    
    all_weight_data[a]['backtest']['pct_return_backtest'] = float(return_backtest)
    all_weight_data[a]['backtest']['std_backtest'] = float(std_backtest)
    all_weight_data[a]['backtest']['sharpe_ratio_backtest'] = float(sharpe_ratio_backtest)
    
    all_weight_data[a]['backtest']['pct_return_spy'] = float(return_spy)
    all_weight_data[a]['backtest']['std_spy'] = float(std_spy)
    all_weight_data[a]['backtest']['sharpe_ratio_spy'] = float(sharpe_ratio_spy)
    
    all_weight_data[a]['backtest']['pct_return_1n'] = float(return_1n)

    all_weight_data[a]['backtest']['pct_std_backtest'] = float(std_backtest)
    all_weight_data[a]['backtest']['high_water_mark'] = mb.high_water_mark
    all_weight_data[a]['backtest']['drawdown'] = mb.drawdown
    all_weight_data[a]['backtest']['stock_money_profit'] = mb.calculate_underperformed_stocks(allocation, df_port)

    print('all_weight_data[a] : ')
    print(all_weight_data[a])

    print('======')

allocation : 
{'KO': 20.0, 'XOM': 20.0, 'LLY': 20.0, 'CVX': 20.0, 'PFE': 16.205332488018538, 'UNH': 3.79466751198151}
start_date_str :  2022-07-01
end_date_str :  2023-07-01
Total Money end :  111.45491570181719
Total Money start :  100.00000000000006
Total Money end :  425.9974365234375
Total Money start :  360.4895935058594
return_backtest :  11.454915701817123
std_backtest :  16.974254363586205
all_weight_data[a] : 
{'expected_annual_return': '45.50520454203638%', 'annual_volatility': '19.19015426258591%', 'sharpe_ratio': 2.1107284489634024, 'weights_to_buy': {'KO': 20.0, 'XOM': 20.0, 'LLY': 20.0, 'CVX': 20.0, 'PFE': 16.205332488018538, 'UNH': 3.79466751198151}, 'backtest': {'start_date_backtest': '2022-07-01', 'end_date_backtest': '2023-07-01', 'pct_return_backtest': 11.454915701817123, 'std_backtest': 16.974254363586205, 'sharpe_ratio_backtest': 0.6748405824759307, 'pct_return_spy': 18.17190959120249, 'std_spy': 19.331596882496996, 'sharpe_ratio_spy': 0.9400107865716723, 'pct_retu

In [12]:
mean_backtest = []
std_backtest = []
sr_backtest = []
mean_spy = []
std_spy = []
sr_spy = []

for a in all_weight_data:
    mean_backtest.append(all_weight_data[a]['backtest']['pct_return_backtest'])
    std_backtest.append(all_weight_data[a]['backtest']['std_backtest'])
    sr_backtest.append(all_weight_data[a]['backtest']['sharpe_ratio_backtest'])

    mean_spy.append(all_weight_data[a]['backtest']['pct_return_spy'])
    std_spy.append(all_weight_data[a]['backtest']['std_spy'])
    sr_spy.append(all_weight_data[a]['backtest']['sharpe_ratio_spy'])

In [13]:
print('backtest MVO: ')
print('mean : ', np.mean(mean_backtest))
print('std : ', np.mean(std_backtest))
print('sr : ', np.mean(sr_backtest))

backtest MVO: 
mean :  32.43162785645742
std :  18.751724312174616
sr :  1.6071698504621157


In [14]:
print('SPY: ')
print('mean : ', np.mean(mean_spy))
print('std : ', np.mean(std_spy))
print('sr : ', np.mean(sr_spy))

SPY: 
mean :  20.96396283639609
std :  15.308588543891249
sr :  1.4896867869982864


In [15]:
px = analyzer.prepare_price_panel(df_pypfopt)
wf = analyzer.walk_forward(px, first_rebalance='2022-07-01', train_years=1, freq='QS',
                           max_weight=0.20, risk_free_rate=0.05, cost_bps=20, benchmark='SPY')

rf_eval = 0.05
wf_summary = pd.DataFrame({
    'MVO gross': analyzer.performance_summary(wf['mvo_gross'], rf_eval),
    'MVO net 20 bps': analyzer.performance_summary(wf['mvo_net'], rf_eval),
    'SPY': analyzer.performance_summary(wf['benchmark'], rf_eval),
    '1/N same universe': analyzer.performance_summary(wf['equal_weight'], rf_eval)
}).T

print('out-of-sample : ', wf['mvo_gross'].index[0].date(), 'to', wf['mvo_gross'].index[-1].date())
print('average in-sample (optimizer) sharpe : ', round(float(np.mean(wf['in_sample_sharpe'])), 2))
wf_summary
      

out-of-sample :  2022-07-05 to 2026-10-16
average in-sample (optimizer) sharpe :  5.32


,cagr,volatility,sharpe_ratio,max_drawdown,growth_of_1
MVO gross,0.438156,0.200452,1.664515,-0.166275,4.671141
MVO net 20 bps,0.428126,0.200580,1.628644,-0.167445,4.534506
SPY,0.187771,0.160818,0.839453,-0.188235,2.075013
1/N same universe,0.233118,0.163392,1.058418,-0.195224,2.432466
